# Alpecca Colab T4 Fast-Brain Server

Run this notebook in Google Colab with a T4 GPU runtime. It exposes an OpenAI-compatible `/v1/chat/completions` endpoint through a Cloudflare quick tunnel. Paste the printed `ALPECCA_COLAB_URL` into your local Alpecca environment.

In [ ]:
!pip -q install fastapi uvicorn transformers accelerate bitsandbytes sentencepiece cloudpickle
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [ ]:
import os

# Leave token empty for private throwaway testing. Set a secret token if you
# share the URL beyond your own machine.
os.environ.setdefault('ALPECCA_COLAB_TOKEN', '')
os.environ.setdefault('ALPECCA_COLAB_MODEL', 'Qwen/Qwen2.5-7B-Instruct')
os.environ.setdefault('ALPECCA_MAX_NEW_TOKENS', '180')
os.environ.setdefault('ALPECCA_TEMPERATURE', '0.72')
print('model =', os.environ['ALPECCA_COLAB_MODEL'])

In [ ]:
import threading, time, os
from typing import Any

import torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import uvicorn

MODEL_ID = os.environ['ALPECCA_COLAB_MODEL']
TOKEN = os.environ.get('ALPECCA_COLAB_TOKEN', '')
MAX_NEW_TOKENS = int(os.environ.get('ALPECCA_MAX_NEW_TOKENS', '180'))
TEMPERATURE = float(os.environ.get('ALPECCA_TEMPERATURE', '0.72'))

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    quantization_config=bnb,
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()

app = FastAPI(title='Alpecca Colab T4 Fast Brain')

class ChatRequest(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]]
    max_tokens: int | None = None
    temperature: float | None = None
    stream: bool | None = False

def check_auth(authorization: str | None):
    if not TOKEN:
        return
    if authorization != f'Bearer {TOKEN}':
        raise HTTPException(status_code=401, detail='bad token')

@app.get('/health')
def health():
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    return {'ready': True, 'backend': 'colab-t4', 'model': MODEL_ID, 'gpu': gpu}

@app.get('/v1/models')
def models():
    return {'object': 'list', 'data': [{'id': MODEL_ID, 'object': 'model'}]}

@app.post('/v1/chat/completions')
def chat(req: ChatRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    messages = req.messages or []
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    max_new = min(int(req.max_tokens or MAX_NEW_TOKENS), 240)
    temp = float(req.temperature or TEMPERATURE)
    started = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=temp > 0,
            temperature=max(0.01, temp),
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    return {
        'id': 'alpecca-colab',
        'object': 'chat.completion',
        'model': MODEL_ID,
        'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': text}, 'finish_reason': 'stop'}],
        'usage': {'completion_seconds': round(time.perf_counter() - started, 3)},
    }

def run_server():
    uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)
print('local server ready: http://127.0.0.1:8000/health')

In [ ]:
import subprocess, re, time

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
print('waiting for Cloudflare tunnel...')
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-zA-Z0-9]+\.trycloudflare\.com', line)
    if m:
        print('\nCOPY THIS INTO YOUR LOCAL ALPECCA SHELL:')
        print('ALPECCA_COLAB_URL=' + m.group(0))
        if os.environ.get('ALPECCA_COLAB_TOKEN'):
            print('ALPECCA_COLAB_API_KEY=' + os.environ['ALPECCA_COLAB_TOKEN'])
        break